# Understanding and Developing Large Language Models
## Laboratory 1: Self-hosted SLMs, model clients, and structured output

A language model returns text across a software boundary. Before an application can use that text, it must establish whether the response has the required structure and meaning.

In this laboratory, we first self-host a small language model (SLM) with LM Studio and expose it through a local HTTP API. We then build a component that classifies messages sent by students to a university course. A deterministic backend remains available for repeatable tests.

By the end of the laboratory, you should be able to:

1. explain the roles of model weights, an inference server, and an API client;
2. run an instruction-tuned SLM locally and connect to it from Python;
3. distinguish raw model output from application data;
4. define and enforce an output contract;
5. handle malformed model responses;
6. evaluate the component against a fixed dataset and a simple baseline.

### 0. Python environment

Run the following cell once. It installs the shared laboratory package and the test dependencies in the active notebook kernel.

In [2]:
from pathlib import Path

LABS_ROOT = Path.cwd().resolve()
if not (LABS_ROOT / "pyproject.toml").exists():
    LABS_ROOT = LABS_ROOT.parent

assert (LABS_ROOT / "pyproject.toml").exists(), "Open this notebook from the Labs_2026 folder."
installation_target = f"{LABS_ROOT}[dev]"
%pip install -q -e "$installation_target"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.2.1
[notice] To update, run: C:\Users\David\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import json
from typing import Literal

import httpx
from IPython.display import JSON, display
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from udllm_labkit import (
    FaultyBackend,
    MockBackend,
    ModelSettings,
    OpenAICompatibleBackend,
    StructuredModel,
)

print(f"Laboratory root: {LABS_ROOT}")

Laboratory root: C:\Users\David\Desktop\LLM-Labs


### 1. Self-host a small language model with LM Studio

A local model application has four distinct parts:

```text
Jupyter notebook  --HTTP-->  LM Studio server  -->  inference runtime  -->  model weights
     client                    localhost:1234            CPU/GPU             local disk/RAM
```

The model weights do not execute themselves. LM Studio loads the weights into memory, performs inference, and exposes an API. The notebook is an API client. There is no universally agreed parameter boundary for an SLM; in this laboratory the term refers operationally to an instruction-tuned model small enough to run on a student computer.

Complete this setup before continuing:

1. Install [LM Studio](https://lmstudio.ai/download) and confirm that your computer satisfies its [system requirements](https://lmstudio.ai/docs/app/system-requirements).
2. In **Discover**, search for `Qwen2.5-7B-Instruct-GGUF`. Choose the `Q4_K_M` quantization. If the machine cannot accommodate it, use a smaller instruction-tuned model and record the substitution.
3. In **Developer**, load the model and start the local server on port `1234`. Leave authentication disabled for this localhost exercise.
4. Keep LM Studio open while using this notebook.

The model is stored and executed locally. The notebook will communicate with it through LM Studio's OpenAI-compatible API.

#### Checkpoint: can the notebook see the server and a model?

The `/v1/models` endpoint reports the model identifiers visible to the server. Run this check before debugging prompts or schemas.

In [4]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
model_ids = []

try:
    response = httpx.get(f"{LM_STUDIO_BASE_URL}/models", timeout=5.0)
    response.raise_for_status()
    model_ids = [item["id"] for item in response.json().get("data", [])]
except (httpx.HTTPError, ValueError, KeyError) as error:
    print(f"LM Studio checkpoint not passed: {error}")

if model_ids:
    print("Models visible to LM Studio:")
    for model_id in model_ids:
        print(f"  - {model_id}")
else:
    print("Start the server and make a downloaded model available, then run this cell again.")

Models visible to LM Studio:
  - qwen2.5-7b-instruct
  - text-embedding-nomic-embed-text-v1.5


Select the exact identifier printed above. If several models are listed, replace the expression with the identifier chosen for this laboratory. The notebook uses the deterministic test double until this checkpoint passes; this keeps the remaining software exercises runnable but does **not** satisfy the local-inference task.

In [6]:
MODEL_NAME = next(
    (model_id for model_id in model_ids if "embedding" not in model_id.lower()),
    None,
)

if MODEL_NAME is not None:
    live_settings = ModelSettings(
        backend="openai-compatible",
        base_url=LM_STUDIO_BASE_URL,
        model_name=MODEL_NAME,
        api_key="lm-studio",
        timeout_seconds=120.0,
    )
    active_backend = OpenAICompatibleBackend(live_settings)
    print(f"Using the self-hosted model: {MODEL_NAME}")
else:
    active_backend = MockBackend()
    print("Using the deterministic test double. Return to the LM Studio checkpoint.")

Using the self-hosted model: qwen2.5-7b-instruct


### 2. Define the output contract

The classifier must return four fields. `Literal` restricts categorical values, `Field` constrains the urgency range, and `extra="forbid"` rejects fields that the application did not request.

In [7]:
class StudentRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    intent: Literal["course_question", "administrative", "technical", "other"]
    urgency: int = Field(ge=1, le=5)
    language: Literal["en", "ro", "other"]
    requires_human: bool

display(JSON(StudentRequest.model_json_schema()))

<IPython.core.display.JSON object>

The schema is an executable contract. Try validating one acceptable response and one unacceptable response.

In [8]:
valid_response = {
    "intent": "technical",
    "urgency": 4,
    "language": "en",
    "requires_human": False,
}

request = StudentRequest.model_validate(valid_response)
print(request.model_dump_json(indent=2))

invalid_response = valid_response | {"urgency": 8}
try:
    StudentRequest.model_validate(invalid_response)
except ValidationError as error:
    print(error)

{
  "intent": "technical",
  "urgency": 4,
  "language": "en",
  "requires_human": false
}
1 validation error for StudentRequest
urgency
  Input should be less than or equal to 5 [type=less_than_equal, input_value=8, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


### 3. Inspect a raw model response

A model backend accepts messages and returns a response. When the LM Studio checkpoint has passed, this cell sends the request to the model running on your computer. Otherwise it uses a deterministic test double so that the software path remains inspectable.

In [9]:
SYSTEM_PROMPT = """Classify a student request for a university course.
Return only an object that satisfies the supplied schema.
Urgency uses a scale from 1 (routine) to 5 (immediate obstruction).
Set requires_human for appeals, safety concerns, personal cases, or an explicit
request to speak with a member of staff.
"""

student_message = "My local model will not run and I am blocked today."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": f"<TASK>classify_student_request</TASK>\nSTUDENT_REQUEST: {student_message}",
    },
]

raw_reply = active_backend.complete(messages, json_schema=StudentRequest.model_json_schema())

print(type(raw_reply.text).__name__)
print(raw_reply.text)
print(f"model={raw_reply.model} latency_ms={raw_reply.latency_ms:.3f}")

HTTPStatusError: Client error '400 Bad Request' for url 'http://localhost:1234/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

### 4. Validate at the application boundary

The raw response is a string. `StructuredModel` asks the backend for a schema-shaped response and validates it before returning a `StudentRequest`. Application code should use the validated object, not the original string.

In [10]:
structured_model = StructuredModel(active_backend)
validated_request, reply = structured_model.generate(messages, StudentRequest)

print(type(validated_request).__name__)
print(validated_request.model_dump_json(indent=2))
print(f"input_tokens={reply.input_tokens} output_tokens={reply.output_tokens}")

HTTPStatusError: Client error '400 Bad Request' for url 'http://localhost:1234/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

Model output is untrusted input. The next cell injects invalid JSON and confirms that it does not cross the validation boundary.

In [11]:
faulty_model = StructuredModel(FaultyBackend(mode="invalid_json"))

try:
    faulty_model.generate(messages, StudentRequest)
except ValueError as error:
    print(f"Rejected response: {error}")

Rejected response: Invalid structured model response: Model output does not contain a JSON object


### Exercise 1 — Probe the contract

Create at least five inputs that cover Romanian and English, routine and urgent requests, and a request that requires human review. Before running the cell, write down the expected result for each input. Record any disagreement between your expectation and the component.

In [12]:
examples = [
    "When is the project deadline?",
    "Notebook-ul produce o eroare Python.",
    "I want to submit a grade appeal.",
    "Ce subiecte sunt discutate la laborator?",
    "Thank you for the explanation.",
]

for text in examples:
    current_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"<TASK>classify_student_request</TASK>\nSTUDENT_REQUEST: {text}"},
    ]
    result, _ = structured_model.generate(current_messages, StudentRequest)
    print(f"{text}\n  -> {result.model_dump()}\n")

HTTPStatusError: Client error '400 Bad Request' for url 'http://localhost:1234/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

### 5. Use the reusable application component

The laboratory package separates the schema, prompting, backend, and metrics from notebook presentation. This makes the same component available to tests, command-line evaluation, and later applications.

In [11]:
from lab01_model_contracts.classifier import classify_student_request

classification = classify_student_request(
    "Please let me speak to the teacher about a personal situation.",
    backend=active_backend,
)

print(classification.request.model_dump_json(indent=2))
print(
    f"model={classification.model} "
    f"tokens={classification.input_tokens + classification.output_tokens} "
    f"latency_ms={classification.latency_ms:.3f}"
)

{
  "intent": "administrative",
  "urgency": 4,
  "language": "en",
  "requires_human": true
}
model=google/gemma-4-12b tokens=141 latency_ms=3006.087


### 6. Evaluate a fixed dataset

A few successful examples do not establish quality. We therefore evaluate the component on a committed JSON Lines dataset. Keeping the data in the repository makes the result inspectable and reproducible. We run the same evaluation twice: first with the deterministic backend to verify the pipeline, then with the SLM served by LM Studio to measure the actual model.

In [12]:
DATA_PATH = LABS_ROOT / "lab01_model_contracts" / "data" / "requests.jsonl"
dataset = [
    json.loads(line)
    for line in DATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Loaded {len(dataset)} labelled messages.")
display(dataset[:3])

Loaded 10 labelled messages.


[{'text': 'When is the project deadline?',
  'intent': 'course_question',
  'requires_human': False},
 {'text': 'The notebook produces a Python import error.',
  'intent': 'technical',
  'requires_human': False},
 {'text': 'I need a certificate confirming my enrolment.',
  'intent': 'administrative',
  'requires_human': False}]

In [13]:
from lab01_model_contracts.evaluate import evaluate

evaluation_results = {
    "deterministic_test_double": evaluate(
        DATA_PATH,
        backend=MockBackend(),
    )
}

if MODEL_NAME is not None:
    evaluation_results[f"lm_studio:{MODEL_NAME}"] = evaluate(
        DATA_PATH,
        backend=active_backend,
    )
else:
    print("LM Studio evaluation skipped: return to the server checkpoint.")

display(evaluation_results)

{'deterministic_test_double': {'examples': 10.0,
  'intent_accuracy': 1.0,
  'review_accuracy': 1.0,
  'mean_latency_ms': 0.02029971219599247,
  'mean_tokens': 120.6},
 'lm_studio:google/gemma-4-12b': {'examples': 10.0,
  'intent_accuracy': 0.8,
  'review_accuracy': 0.9,
  'mean_latency_ms': 3266.1303001455963,
  'mean_tokens': 139.6}}

### 7. Compare with a baseline

A system should justify the use of an LLM against a simpler alternative. The baseline below deliberately uses few rules. Measure it first; then inspect its errors before adding more rules.

In [14]:
def keyword_baseline(text: str) -> str:
    lowered = text.lower()
    if any(word in lowered for word in ("error", "install", "python", "run")):
        return "technical"
    if any(word in lowered for word in ("grade", "absence", "absent", "certificate", "enrol")):
        return "administrative"
    if any(word in lowered for word in ("lecture", "lab", "laborator", "deadline", "course", "exam", "project")):
        return "course_question"
    return "other"

baseline_predictions = [keyword_baseline(row["text"]) for row in dataset]
baseline_accuracy = sum(
    prediction == row["intent"]
    for prediction, row in zip(baseline_predictions, dataset, strict=True)
) / len(dataset)

print(f"Baseline intent accuracy: {baseline_accuracy:.1%}")
for prediction, row in zip(baseline_predictions, dataset, strict=True):
    if prediction != row["intent"]:
        print(f"Expected {row['intent']!r}, predicted {prediction!r}: {row['text']}")

Baseline intent accuracy: 100.0%


### Exercise 2 — Extend and test the system

Complete the following changes in the package files, not only in this notebook:

1. Add a `feedback` intent to `schemas.py` and update the classifier examples.
2. Add five difficult, labelled messages to `data/requests.jsonl`. Include at least one ambiguous message.
3. Move the keyword baseline into a separate function in the package.
4. Add one test for malformed JSON and one test for a schema-valid but semantically incorrect response.

In [15]:
%cd $LABS_ROOT
!python -m pytest -q lab01_model_contracts/tests

/Users/b.mursa/Facultate/Materials/UDLLM/Labs_2026
.....                                                                    [100%]Running teardown with pytest sessionfinish...

5 passed in 0.01s
